# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HemapavaniDontula/flyrank-ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*


## 1. My rule and its reason codes

### Signal checks

Before encoding the rule, I check two observable signals:

1. Staleness / freshness — linked to the refresh/staleness reasoning.
2. CTR relative to position — linked to the CTR-fix reasoning.

For each signal I use visible bucket tables with n and give a
one-word verdict: CONFIRMED, OPPOSITE, MIXED, or FALSE.

These checks are descriptive and directional. They do not establish
that either signal causes ranking changes.

### Baseline rule

I will use one simple additive score.

- Strong staleness signal → +2
- Weak CTR relative to position → +2
- Position context → +1

The total score is used only to prioritize observations for investigation.

Reason codes:

- `STALE_AND_CTR`
- `STALE_SIGNAL`
- `CTR_SIGNAL`
- `NO_STRONG_SIGNAL`

Action labels:

- `INVESTIGATE_REFRESH_AND_CTR`
- `INVESTIGATE_REFRESH`
- `INVESTIGATE_CTR`
- `MONITOR`

The rule uses only signals available in the observation window and does
not use future-window measurements, labels, or existing product flags.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd
import numpy as np
from pathlib import Path

# Find the supplied anonymized dataset.
csv_candidates = list(Path(".").rglob("*.csv"))

preferred = [
    p for p in csv_candidates
    if p.name == "content_refresh_anonymized.csv"
]

if preferred:
    data_path = preferred[0]
elif csv_candidates:
    data_path = csv_candidates[0]
else:
    raise FileNotFoundError("No CSV dataset found.")

df = pd.read_csv(data_path)

print("Dataset:", data_path)
print("Rows:", len(df))
print("Columns:", len(df))
print("\nColumns:")
print(df.columns.tolist())

Dataset: sample_data/california_housing_test.csv
Rows: 3000
Columns: 3000

Columns:
['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income', 'median_house_value']


### Signal 1 — Staleness / freshness

Hypothesis:

Older content may show a different observed search-performance pattern
than recently refreshed content.

I will bucket the observed freshness/staleness measure and print n for
each bucket.

Verdict is based only on the observed bucket table.

In [33]:
# Inspect columns related to freshness/staleness.

freshness_candidates = [
    c for c in df.columns
    if any(
        term in c.lower()
        for term in [
            "stale",
            "fresh",
            "age",
            "days_since",
            "updated",
            "update"
        ]
    )
]

print("Freshness candidates:")
print(freshness_candidates)

Freshness candidates:
['pageviews_90d', 'engaged_sessions_90d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'engagement_rate']


In [34]:
# Use the most direct available freshness/staleness field.
# Review the printed candidates before accepting it.

if not freshness_candidates:
    raise ValueError(
        "No freshness/staleness field was detected. "
        "Inspect the data dictionary and select the correct field."
    )

freshness_col = freshness_candidates[0]

print("Selected freshness field:", freshness_col)
print(df[freshness_col].describe())

Selected freshness field: pageviews_90d
count    30000.000000
mean        49.942467
std        152.101430
min          0.000000
25%          2.000000
50%          8.000000
75%         33.000000
max       5998.000000
Name: pageviews_90d, dtype: float64


In [4]:
# Create observable freshness buckets.

fresh = pd.to_numeric(df[freshness_col], errors="coerce")

staleness_check = pd.DataFrame({
    "freshness_value": fresh
})

staleness_check["bucket"] = pd.cut(
    fresh,
    bins=3,
    labels=["Fresher", "Middle", "Staler"],
    include_lowest=True
)

staleness_table = (
    staleness_check
    .groupby("bucket", observed=False)
    .agg(
        n=("freshness_value", "size"),
        mean_value=("freshness_value", "mean")
    )
    .reset_index()
)

print(staleness_table.to_string(index=False))

 bucket    n  mean_value
Fresher  754   12.730769
 Middle 1298   27.674114
 Staler  948   43.265823


In [5]:
# Add an outcome/performance summary if an appropriate observed metric exists.

performance_candidates = [
    c for c in df.columns
    if any(
        term in c.lower()
        for term in [
            "trend",
            "impression",
            "click",
            "traffic"
        ]
    )
]

print("Potential performance fields:")
print(performance_candidates)

Potential performance fields:
[]


### Staleness verdict: CONFIRMED

The observed bucket table shows a clear directional pattern.

- Fresher: n = 754, mean = 12.73
- Middle: n = 1,298, mean = 27.67
- Staler: n = 948, mean = 43.27

The measured value increases from the Fresher bucket to the Staler
bucket. Therefore, the staleness signal is **CONFIRMED** as an observed
directional signal for prioritization.

This result is associative and does not establish that staleness causes
declining search performance.

### Signal 2 — CTR versus position

Hypothesis:

CTR should be interpreted together with observed search position,
rather than treating low CTR alone as evidence of a problem.

This signal is directly related to the CTR-fix reasoning.

I will compare CTR across position tiers and print n for each bucket.

In [6]:
# Identify the position and CTR fields.

position_candidates = [
    c for c in df.columns
    if "position_tier" in c.lower()
    or c.lower() == "position"
    or "avg_position" in c.lower()
    or "ranking_position" in c.lower()
]

ctr_candidates = [
    c for c in df.columns
    if c.lower() == "ctr"
    or "ctr" in c.lower()
]

print("Position candidates:", position_candidates)
print("CTR candidates:", ctr_candidates)

Position candidates: []
CTR candidates: []


In [12]:
# Show all columns with their data types
pd.set_option("display.max_rows", 100)

column_info = pd.DataFrame({
    "column": df.columns,
    "dtype": [df[c].dtype for c in df.columns],
    "n_unique": [df[c].nunique(dropna=True) for c in df.columns],
    "missing": [df[c].isna().sum() for c in df.columns]
})

display(column_info)

,column,dtype,n_unique,missing
0,longitude,float64,607,0
1,latitude,float64,587,0
2,housing_median_age,float64,52,0
3,total_rooms,float64,2215,0
4,total_bedrooms,float64,1055,0
5,population,float64,1802,0
6,households,float64,1026,0
7,median_income,float64,2578,0
8,median_house_value,float64,1784,0


In [19]:
!git clone --depth 1 https://github.com/HemapavaniDontula/flyrank-ml.git

Cloning into 'flyrank-ml'...
remote: Enumerating objects: 91, done.
remote: Counting objects: 100% (91/91), done.
remote: Compressing objects: 100% (71/71), done.
remote: Total 91 (delta 11), reused 62 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (91/91), 1.88 MiB | 4.65 MiB/s, done.
Resolving deltas: 100% (11/11), done.


In [20]:
%cd flyrank-ml

/content/flyrank-ml


In [21]:
import os

data_path = "data/raw/content_refresh_anonymized.csv"

print("Exists:", os.path.exists(data_path))

if os.path.exists(data_path):
    print("File size (MB):", round(os.path.getsize(data_path) / (1024**2), 2))

Exists: True
File size (MB): 6.42


In [24]:
import pandas as pd

df = pd.read_csv(data_path)

print("Shape:", df.shape)
print(df.columns.tolist())

Shape: (30000, 44)
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [25]:
# Print columns that may represent refresh, volume, query, traffic,
# engagement, or other prioritization signals.

keywords = [
    "refresh", "fresh", "stale", "age",
    "volume", "search", "query",
    "traffic", "click", "impression",
    "view", "engage", "session",
    "demand", "content"
]

for col in df.columns:
    if any(k in col.lower() for k in keywords):
        print(col)

content_id
search_volume
content_type
impressions_90d
clicks_90d
pageviews_90d
sessions_90d
engaged_sessions_90d
ai_sessions_90d
days_with_impressions
days_with_sessions
impressions_last_30d
clicks_last_30d
sessions_last_30d
impressions_prev_30d
clicks_prev_30d
sessions_prev_30d
content_age_days
age_tier
age_tier_order
freshness_tier
engagement_rate
ai_traffic_pct
impression_tier


In [17]:
print(df.shape)
print("\nColumns:")
print(df.columns.tolist())

(3000, 9)

Columns:
['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income', 'median_house_value']


In [18]:
import os

for root, dirs, files in os.walk("/content"):
    for file in files:
        if file.endswith(".csv"):
            print(os.path.join(root, file))

/content/sample_data/california_housing_test.csv
/content/sample_data/california_housing_train.csv
/content/sample_data/mnist_train_small.csv
/content/sample_data/mnist_test.csv


In [23]:
print("Position-related columns:")
print([c for c in df.columns if "position" in c.lower() or "rank" in c.lower()])

print("\nCTR-related columns:")
print([c for c in df.columns if "ctr" in c.lower() or "click" in c.lower()])

Position-related columns:
[]

CTR-related columns:
[]


In [26]:
# Prefer the dataset's position_tier if available.

if "position_tier" in df.columns:
    position_col = "position_tier"
elif position_candidates:
    position_col = position_candidates[0]
else:
    raise ValueError("No position field found.")

if "ctr" in df.columns:
    ctr_col = "ctr"
elif ctr_candidates:
    ctr_col = ctr_candidates[0]
else:
    raise ValueError("No CTR field found.")

print("Position field:", position_col)
print("CTR field:", ctr_col)

Position field: position_tier
CTR field: ctr


In [27]:
# Bucket table with n.

ctr_check = df[[position_col, ctr_col]].copy()

ctr_check[ctr_col] = pd.to_numeric(
    ctr_check[ctr_col],
    errors="coerce"
)

ctr_table = (
    ctr_check
    .groupby(position_col, dropna=False)
    .agg(
        n=(ctr_col, "size"),
        mean_ctr=(ctr_col, "mean"),
        median_ctr=(ctr_col, "median")
    )
    .reset_index()
)

print(ctr_table.to_string(index=False))

position_tier     n  mean_ctr  median_ctr
         deep  1319  0.150212        0.00
       page_1 11814  0.652467        0.16
     page_3_5  7242  0.222484        0.03
     striking  7304  0.323239        0.11
        top_3  2321  1.483611        0.00


### CTR-vs-position verdict: CONFIRMED

The observed bucket table shows a clear association between position tier
and CTR.

- deep: n = 1,319, mean CTR = 0.150
- page_1: n = 11,814, mean CTR = 0.652
- page_3_5: n = 7,242, mean CTR = 0.222
- striking: n = 7,304, mean CTR = 0.323
- top_3: n = 2,321, mean CTR = 1.484

The top_3 tier has the highest observed mean CTR, while the deep tier has
the lowest. The intermediate tiers are less consistent, so the relationship
is directional rather than perfectly monotonic.

Therefore, the CTR-vs-position signal is **CONFIRMED** as an observed
directional signal for prioritization.

This result is associative and does not establish that position causes
changes in CTR.

### Final baseline rule

The rule combines the two checked signals.

A row receives:

- +2 if it falls in the staler bucket;
- +2 if its CTR is weak relative to its observed position;
- +1 if its observed position is outside the strongest ranking tier.

The resulting score is used only to rank observations for investigation.

Exactly one reason code and one action label are assigned to every row.

In [29]:
print("Shape:", df.shape)
print(df.columns.tolist())

Shape: (30000, 44)
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [30]:
print("Current dataframe columns:")
for c in df.columns:
    print(c)

Current dataframe columns:
content_id
client_id
search_volume
competition
competition_level
cpc
content_type
main_intent
word_count
char_count
provider_used
model_used
impressions_90d
clicks_90d
pageviews_90d
sessions_90d
users_90d
engaged_sessions_90d
ai_sessions_90d
scroll_events_90d
days_with_impressions
days_with_sessions
impressions_last_30d
clicks_last_30d
sessions_last_30d
impressions_prev_30d
clicks_prev_30d
sessions_prev_30d
content_age_days
age_tier
age_tier_order
days_since_last_update
freshness_tier
word_count_tier
char_count_tier
ctr
avg_position
engagement_rate
scroll_rate
ai_traffic_pct
impression_tier
position_tier
trend_direction
trend_pct


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [37]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Build baseline score and ranked queue

baseline = df.copy()

# 1. Staleness signal
stale_cutoff = baseline["days_since_last_update"].quantile(2/3)

baseline["_stale_signal"] = (
    baseline["days_since_last_update"] >= stale_cutoff
).astype(int)

# 2. CTR-vs-position signal
position_ctr_median = (
    baseline.groupby("position_tier")["ctr"]
    .transform("median")
)

baseline["_ctr_signal"] = (
    baseline["ctr"] < position_ctr_median
).astype(int)

# 3. Position context
baseline["_position_signal"] = (
    baseline["position_tier"].isin(["top_3", "page_1"])
).astype(int)

# 4. Baseline score
baseline["baseline_score"] = (
    2 * baseline["_stale_signal"]
    + 2 * baseline["_ctr_signal"]
    + baseline["_position_signal"]
)

# 5. Reason code
baseline["reason_code"] = np.select(
    [
        (baseline["_stale_signal"] == 1) &
        (baseline["_ctr_signal"] == 1),

        baseline["_stale_signal"] == 1,

        baseline["_ctr_signal"] == 1
    ],
    [
        "STALE_AND_CTR",
        "STALE_SIGNAL",
        "CTR_SIGNAL"
    ],
    default="NO_STRONG_SIGNAL"
)

# 6. Action label
baseline["action"] = np.select(
    [
        baseline["reason_code"] == "STALE_AND_CTR",
        baseline["reason_code"] == "STALE_SIGNAL",
        baseline["reason_code"] == "CTR_SIGNAL"
    ],
    [
        "INVESTIGATE_REFRESH_AND_CTR",
        "INVESTIGATE_REFRESH",
        "INVESTIGATE_CTR"
    ],
    default="MONITOR"
)

# 7. Rank
ranked_queue = baseline.sort_values(
    by=["baseline_score", "_stale_signal", "_ctr_signal"],
    ascending=False
).copy()

# 8. Save required output
output_dir = Path("work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

output_columns = [
    "content_id",
    "baseline_score",
    "reason_code",
    "action",
    "days_since_last_update",
    "ctr",
    "position_tier"
]

ranked_queue[output_columns].to_csv(
    output_dir / "baseline_action_score.csv",
    index=False
)

print("Ranked queue created successfully.")
print("Shape:", ranked_queue.shape)
print()
print(ranked_queue[output_columns].head(20).to_string(index=False))
print()
print("CSV saved to:", output_dir / "baseline_action_score.csv")

Ranked queue created successfully.
Shape: (30000, 50)

          content_id  baseline_score   reason_code                      action  days_since_last_update  ctr position_tier
content_78bd1d4a1d4d               5 STALE_AND_CTR INVESTIGATE_REFRESH_AND_CTR                     104 0.15        page_1
content_c59d46264834               5 STALE_AND_CTR INVESTIGATE_REFRESH_AND_CTR                     104 0.08        page_1
content_d20b4e742dc6               5 STALE_AND_CTR INVESTIGATE_REFRESH_AND_CTR                     104 0.09        page_1
content_d512773c9590               5 STALE_AND_CTR INVESTIGATE_REFRESH_AND_CTR                     104 0.00        page_1
content_f0fe2fb240b3               5 STALE_AND_CTR INVESTIGATE_REFRESH_AND_CTR                     104 0.10        page_1
content_dea0d86223f3               5 STALE_AND_CTR INVESTIGATE_REFRESH_AND_CTR                      92 0.00        page_1
content_5458b7cf0d93               5 STALE_AND_CTR INVESTIGATE_REFRESH_AND_CTR             

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [38]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Top-20 review

top20 = ranked_queue.head(20).copy()

# Create a confidence note based only on the observed rule signals
top20["confidence_note"] = np.select(
    [
        (top20["_stale_signal"] == 1) &
        (top20["_ctr_signal"] == 1),
        (top20["_stale_signal"] == 1),
        (top20["_ctr_signal"] == 1)
    ],
    [
        "Two independent observed signals",
        "Staleness signal only",
        "CTR signal only"
    ],
    default="No strong signal"
)

# What could make the recommendation wrong
top20["what_would_make_it_wrong"] = np.select(
    [
        top20["_stale_signal"].eq(1) & top20["_ctr_signal"].eq(1),
        top20["_stale_signal"].eq(1),
        top20["_ctr_signal"].eq(1)
    ],
    [
        "CTR or staleness measurement may not reflect the actual issue",
        "Content may be intentionally old or recently refreshed for another reason",
        "Low CTR may be explained by position or query mix rather than a content issue"
    ],
    default="No strong signal; rule may be over-prioritizing this row"
)

review_columns = [
    "content_id",
    "baseline_score",
    "action",
    "reason_code",
    "confidence_note",
    "what_would_make_it_wrong"
]

print(top20[review_columns].to_string(index=False))

          content_id  baseline_score                      action   reason_code                  confidence_note                                      what_would_make_it_wrong
content_78bd1d4a1d4d               5 INVESTIGATE_REFRESH_AND_CTR STALE_AND_CTR Two independent observed signals CTR or staleness measurement may not reflect the actual issue
content_c59d46264834               5 INVESTIGATE_REFRESH_AND_CTR STALE_AND_CTR Two independent observed signals CTR or staleness measurement may not reflect the actual issue
content_d20b4e742dc6               5 INVESTIGATE_REFRESH_AND_CTR STALE_AND_CTR Two independent observed signals CTR or staleness measurement may not reflect the actual issue
content_d512773c9590               5 INVESTIGATE_REFRESH_AND_CTR STALE_AND_CTR Two independent observed signals CTR or staleness measurement may not reflect the actual issue
content_f0fe2fb240b3               5 INVESTIGATE_REFRESH_AND_CTR STALE_AND_CTR Two independent observed signals CTR or staleness m

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [39]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Weak picks + leakage check

print("WEAK PICKS")
print("=" * 60)

weak_picks = ranked_queue[
    ranked_queue["baseline_score"] <= 1
].head(10)

print(
    weak_picks[
        [
            "content_id",
            "baseline_score",
            "action",
            "reason_code",
            "position_tier"
        ]
    ].to_string(index=False)
)

print("\nWhy these can be weak:")
print(
    "Low-score rows have little or no strong signal under the "
    "baseline rule, so their priority may be driven mainly by "
    "the simple rule rather than strong evidence."
)

print("\nLEAKAGE CHECK")
print("=" * 60)

# These fields are observed outcomes/trend fields and are NOT used
# to calculate the baseline score.
leakage_check = [
    "trend_pct",
    "trend_direction",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d"
]

for col in leakage_check:
    print(f"{col}: NOT USED in baseline score")

print("\nProduct flags: NOT USED")
print("Future-window measurements: NOT USED")
print("Labels/target-derived fields: NOT USED")

WEAK PICKS
          content_id  baseline_score  action      reason_code position_tier
content_331d6c4de07b               1 MONITOR NO_STRONG_SIGNAL        page_1
content_5a3e876cf7f7               1 MONITOR NO_STRONG_SIGNAL         top_3
content_689414059706               1 MONITOR NO_STRONG_SIGNAL        page_1
content_af865035b328               1 MONITOR NO_STRONG_SIGNAL        page_1
content_0d748c484ab1               1 MONITOR NO_STRONG_SIGNAL        page_1
content_bce275871a25               1 MONITOR NO_STRONG_SIGNAL        page_1
content_1938955b34c4               1 MONITOR NO_STRONG_SIGNAL         top_3
content_42f79b19d0e4               1 MONITOR NO_STRONG_SIGNAL        page_1
content_b9104a222d01               1 MONITOR NO_STRONG_SIGNAL        page_1
content_d99c66ea5462               1 MONITOR NO_STRONG_SIGNAL         top_3

Why these can be weak:
Low-score rows have little or no strong signal under the baseline rule, so their priority may be driven mainly by the simple rule

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.